# IDX Next-Day Price Direction Prediction
## Model Training & Evaluation

This notebook is intentionally separate from the EDA and feature-engineering notebooks. It loads the already exported feature dataset from `Dataset/processed`, reconstructs the next trading date from `Dataset/cleaned`, applies a strictly temporal split, trains classifiers, and evaluates them with metrics suitable for an imbalanced binary classification task.

Prediction timing assumption: features observed on day `t` are used after market close on day `t` to predict whether `Close(t+1) > Close(t)`. Logistic Regression is retained as an interpretable baseline; Random Forest is the selected final model because it performs better on the validation/test ranking and correlation metrics.

---
## Section 1: Setup

In [76]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    matthews_corrcoef,
    precision_score,
    PrecisionRecallDisplay,
    recall_score,
    RocCurveDisplay,
    roc_auc_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

DATA_DIR = Path('Dataset')
PROCESSED_DIR = DATA_DIR / 'processed'
CLEANED_DIR = DATA_DIR / 'cleaned'
MODEL_OUTPUT_DIR = DATA_DIR / 'model_outputs'
MODEL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Processed dir: {PROCESSED_DIR.resolve()}')
print(f'Cleaned dir:   {CLEANED_DIR.resolve()}')
print(f'Output dir:    {MODEL_OUTPUT_DIR.resolve()}')

Processed dir: C:\Users\Rafi\OneDrive\Documents\CSUI24\Sem 4\Kasdad\miniproject\JCI-Time-Series\Dataset\processed
Cleaned dir:   C:\Users\Rafi\OneDrive\Documents\CSUI24\Sem 4\Kasdad\miniproject\JCI-Time-Series\Dataset\cleaned
Output dir:    C:\Users\Rafi\OneDrive\Documents\CSUI24\Sem 4\Kasdad\miniproject\JCI-Time-Series\Dataset\model_outputs


---
## Section 2: Load Processed Modeling Dataset

The processed parquet files contain engineered features and the binary target. If the files do not include a `sector` column, the sector is restored from the filename.

In [77]:
SECTOR_FILES = {
    'Energy': PROCESSED_DIR / 'energy.parquet',
    'Technology': PROCESSED_DIR / 'technology.parquet',
}

frames = []
for sector, path in SECTOR_FILES.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing processed file: {path}')
    part = pd.read_parquet(path)
    if 'sector' not in part.columns:
        part['sector'] = sector
    frames.append(part)

model_df = pd.concat(frames, ignore_index=True)
model_df['Date'] = pd.to_datetime(model_df['Date'])
model_df['target'] = model_df['target'].astype(int)
model_df = model_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

print(f'Rows:          {len(model_df):,}')
print(f'Tickers:       {model_df["Ticker"].nunique():,}')
print(f'Date range:    {model_df["Date"].min().date()} to {model_df["Date"].max().date()}')
print('Sector rows:')
print(model_df['sector'].value_counts())
print('Target balance:')
print(model_df['target'].value_counts(normalize=True).sort_index().rename({0: 'Down/Flat', 1: 'Up'}))

Rows:          84,051
Tickers:       126
Date range:    2020-03-12 to 2024-05-16
Sector rows:
sector
Energy        58418
Technology    25633
Name: count, dtype: int64
Target balance:
target
Down/Flat    0.625097
Up           0.374903
Name: proportion, dtype: float64


---
## Section 2B: Ticker Universe Reduction

Topic C limits the modeling universe to at most 100 stocks across 2-3 IDX-IC sectors. To make this notebook compliant and reproducible while using the full cap, the model keeps 56 Energy tickers and 44 Technology tickers. Tickers are ranked within each sector by processed row count, then median volume, then average volume.

The processed dataset currently contains 44 Technology tickers after cleaning and feature-engineering filters, so all available Technology tickers are retained. Energy is capped at 56 tickers so the final modeling universe is 56 Energy + 44 Technology = 100 tickers.

In [78]:
TICKER_TARGETS = {
    'Energy': 56,
    'Technology': 44,
}

selection_rows = []
selected_tickers = []

for sector, target_n in TICKER_TARGETS.items():
    sector_df = model_df[model_df['sector'] == sector].copy()
    if sector_df.empty:
        raise ValueError(f'No rows found for sector: {sector}')

    ticker_rank = (
        sector_df.groupby('Ticker')
        .agg(
            rows=('Date', 'size'),
            start_date=('Date', 'min'),
            end_date=('Date', 'max'),
            median_volume=('Volume', 'median'),
            avg_volume=('Volume', 'mean'),
            up_rate=('target', 'mean'),
        )
        .reset_index()
        .sort_values(
            ['rows', 'median_volume', 'avg_volume', 'Ticker'],
            ascending=[False, False, False, True],
        )
    )

    available_n = len(ticker_rank)
    selected_n = min(target_n, available_n)
    selected = ticker_rank.head(selected_n).copy()
    selected['sector'] = sector
    selected['target_n'] = target_n
    selected['available_n'] = available_n
    selected['selected_n'] = selected_n

    selection_rows.append(selected)
    selected_tickers.extend(selected['Ticker'].tolist())

    if available_n < target_n:
        print(
            f'{sector}: requested {target_n} tickers, but only {available_n} are available '
            'after cleaning/feature engineering; keeping all available tickers.'
        )
    else:
        print(f'{sector}: selected top {selected_n} of {available_n} available tickers.')

ticker_selection = pd.concat(selection_rows, ignore_index=True)
model_df = model_df[model_df['Ticker'].isin(selected_tickers)].copy()
model_df = model_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

selection_summary = (
    ticker_selection.groupby('sector')
    .agg(
        available_tickers=('available_n', 'first'),
        selected_tickers=('Ticker', 'nunique'),
        target_tickers=('target_n', 'first'),
        selected_rows=('rows', 'sum'),
    )
    .reindex(TICKER_TARGETS.keys())
)

print('Ticker selection summary:')
display(selection_summary)

print(f'Final modeling universe: {model_df["Ticker"].nunique()} tickers, {len(model_df):,} rows')
print('Final sector ticker counts:')
print(model_df.groupby('sector')['Ticker'].nunique())

if model_df['Ticker'].nunique() > 100:
    raise ValueError('Ticker cap violated: selected more than 100 tickers')

print('Top selected tickers by sector:')
display(ticker_selection[['sector', 'Ticker', 'rows', 'median_volume', 'avg_volume', 'up_rate']].head(20))

Energy: selected top 56 of 82 available tickers.
Technology: selected top 44 of 44 available tickers.
Ticker selection summary:


,available_tickers,selected_tickers,target_tickers,selected_rows
sector,,,,
Energy,82,56,56,51086
Technology,44,44,44,25633


Final modeling universe: 100 tickers, 76,719 rows
Final sector ticker counts:
sector
Energy        56
Technology    44
Name: Ticker, dtype: int64
Top selected tickers by sector:


,sector,Ticker,rows,median_volume,avg_volume,up_rate
0,Energy,PGAS,1007,73593700.0,1.093538e+08,0.442900
1,Energy,ELSA,1007,36650400.0,6.833086e+07,0.424032
2,Energy,HRUM,1007,20970800.0,3.805830e+07,0.456802
3,Energy,INDY,1007,13986900.0,2.364508e+07,0.446872
4,Energy,SOCI,1007,8155200.0,1.796817e+07,0.367428
5,Energy,PSSI,1007,968100.0,2.034849e+06,0.397219
6,Energy,MBSS,1007,609100.0,1.607242e+06,0.398213
7,Energy,ADRO,1006,73200950.0,9.217022e+07,0.462227
8,Energy,MEDC,1006,72713550.0,9.372277e+07,0.439364
9,Energy,DOID,1006,37771350.0,7.953848e+07,0.441352


---
## Section 3: Feature Column Selection

Only engineered feature columns are used as model inputs. Metadata, raw OHLCV columns, the target, and split-related columns are excluded from `X`.

In [79]:
FEATURE_REFERENCE_PATH = PROCESSED_DIR / 'feature_reference.csv'

if FEATURE_REFERENCE_PATH.exists():
    feature_ref = pd.read_csv(FEATURE_REFERENCE_PATH)
    if {'column', 'group'}.issubset(feature_ref.columns):
        feature_cols = feature_ref.loc[feature_ref['group'].eq('feature'), 'column'].tolist()
    else:
        feature_cols = []
else:
    feature_ref = pd.DataFrame()
    feature_cols = []

if not feature_cols:
    excluded = {
        'Date', 'target_date', 'split', 'Ticker', 'sector', 'target',
        'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
    }
    feature_cols = [c for c in model_df.columns if c not in excluded]

missing_features = [c for c in feature_cols if c not in model_df.columns]
if missing_features:
    raise ValueError(f'Feature columns missing from model_df: {missing_features[:10]}')

forbidden_inputs = {
    'Date', 'target_date', 'split', 'Ticker', 'sector', 'target',
    'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume',
}
leaky_inputs = sorted(set(feature_cols) & forbidden_inputs)
if leaky_inputs:
    raise ValueError(f'Forbidden columns included in feature set: {leaky_inputs}')

print(f'Feature count: {len(feature_cols)}')
print(feature_cols[:15])

Feature count: 77
['daily_return', 'log_return', 'high_low_spread', 'close_open_gap', 'price_position', 'volume_change', 'log_volume', 'daily_return_lag1', 'daily_return_lag2', 'daily_return_lag3', 'daily_return_lag5', 'daily_return_lag10', 'daily_return_lag20', 'volume_change_lag1', 'volume_change_lag2']


---
## Section 4: Reconstruct Target Date And Verify Label

The split is based on the date being predicted, not merely the feature row date. The current processed dataset may not contain `target_date`, so this notebook reconstructs it from the cleaned OHLCV files.

In [80]:
cleaned_frames = []
for sector, path in {
    'Energy': CLEANED_DIR / 'energy.parquet',
    'Technology': CLEANED_DIR / 'technology.parquet',
}.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing cleaned file: {path}')
    part = pd.read_parquet(path)
    part['sector'] = sector
    cleaned_frames.append(part[['Date', 'Ticker', 'Close', 'sector']])

cleaned_df = pd.concat(cleaned_frames, ignore_index=True)
cleaned_df['Date'] = pd.to_datetime(cleaned_df['Date'])
cleaned_df = cleaned_df.sort_values(['Ticker', 'Date']).reset_index(drop=True)

g = cleaned_df.groupby('Ticker')
cleaned_df['target_date'] = g['Date'].shift(-1)
cleaned_df['next_close'] = g['Close'].shift(-1)
cleaned_df['expected_target'] = np.where(
    cleaned_df['next_close'].isna(),
    np.nan,
    (cleaned_df['next_close'] > cleaned_df['Close']).astype(int),
)

target_lookup = cleaned_df[['Ticker', 'Date', 'target_date', 'expected_target']]
model_df = model_df.drop(columns=[c for c in ['target_date'] if c in model_df.columns])
model_df = model_df.merge(target_lookup, on=['Ticker', 'Date'], how='left')

missing_target_date = model_df['target_date'].isna().sum()
if missing_target_date:
    raise ValueError(f'{missing_target_date:,} model rows are missing target_date')

label_mismatch = (model_df['target'].astype(float) != model_df['expected_target']).sum()
print(f'Missing target_date rows: {missing_target_date:,}')
print(f'Label mismatches vs cleaned OHLCV: {label_mismatch:,}')
if label_mismatch:
    display(model_df.loc[model_df['target'].astype(float) != model_df['expected_target'],
                         ['Date', 'Ticker', 'Close', 'target', 'expected_target']].head())
    raise ValueError('Target labels do not match cleaned OHLCV reconstruction')

model_df = model_df.drop(columns=['expected_target'])

Missing target_date rows: 0
Label mismatches vs cleaned OHLCV: 0


---
## Section 5: Temporal Train/Validation/Test Split

The split uses `target_date` so no training row uses a label from the validation or test period.

In [81]:
TRAIN_END = pd.Timestamp('2023-01-02')
VALID_END = pd.Timestamp('2024-01-02')

model_df['split'] = np.select(
    [
        model_df['target_date'] < TRAIN_END,
        (model_df['target_date'] >= TRAIN_END) & (model_df['target_date'] < VALID_END),
        model_df['target_date'] >= VALID_END,
    ],
    ['train', 'valid', 'test'],
    default='unassigned',
)

if (model_df['split'] == 'unassigned').any():
    raise ValueError('Some rows were not assigned to train/valid/test')

split_summary = (
    model_df.groupby('split')
    .agg(
        rows=('target', 'size'),
        tickers=('Ticker', 'nunique'),
        start_date=('Date', 'min'),
        end_date=('Date', 'max'),
        target_start=('target_date', 'min'),
        target_end=('target_date', 'max'),
        up_rate=('target', 'mean'),
    )
    .reindex(['train', 'valid', 'test'])
)

display(split_summary)

assert model_df.loc[model_df['split'] == 'train', 'target_date'].max() < TRAIN_END
assert model_df.loc[model_df['split'] == 'valid', 'target_date'].min() >= TRAIN_END
assert model_df.loc[model_df['split'] == 'valid', 'target_date'].max() < VALID_END
assert model_df.loc[model_df['split'] == 'test', 'target_date'].min() >= VALID_END

,rows,tickers,start_date,end_date,target_start,target_end,up_rate
split,,,,,,,
train,48888,87,2020-03-12,2022-12-29,2020-03-13,2022-12-30,0.385289
valid,20530,94,2022-12-30,2023-12-28,2023-01-02,2023-12-29,0.372966
test,7301,95,2023-12-29,2024-05-16,2024-01-02,2024-05-17,0.355705


---
## Section 6: Prepare Model Matrices

In [82]:
train_df = model_df[model_df['split'] == 'train'].copy()
valid_df = model_df[model_df['split'] == 'valid'].copy()
test_df = model_df[model_df['split'] == 'test'].copy()

X_train = train_df[feature_cols]
y_train = train_df['target'].astype(int)
X_valid = valid_df[feature_cols]
y_valid = valid_df['target'].astype(int)
X_test = test_df[feature_cols]
y_test = test_df['target'].astype(int)

print(f'X_train: {X_train.shape}, y mean={y_train.mean():.3f}')
print(f'X_valid: {X_valid.shape}, y mean={y_valid.mean():.3f}')
print(f'X_test:  {X_test.shape}, y mean={y_test.mean():.3f}')

if X_train.isna().sum().sum() or X_valid.isna().sum().sum() or X_test.isna().sum().sum():
    print('NaN values found; model pipelines will impute medians fitted on train only.')
else:
    print('No missing feature values found.')

X_train: (48888, 77), y mean=0.385
X_valid: (20530, 77), y mean=0.373
X_test:  (7301, 77), y mean=0.356
No missing feature values found.


---
## Section 6B: Cap Feature Set To 40

The processed dataset exposes 77 candidate model features. To keep the final model compact and easier to explain, this notebook caps the modeling feature set at 40 features. The ranking step is fit on the training period only: a lightweight Random Forest selector learns importances from `X_train`/`y_train`, then validation and test matrices are rebuilt with the selected columns. This avoids using validation or test outcomes when choosing features.

In [ ]:
# Fit the selector on training data only to avoid validation/test leakage.
FEATURE_CAP = 40
all_feature_cols = feature_cols.copy()

if len(all_feature_cols) < FEATURE_CAP:
    raise ValueError(f'Feature cap ({FEATURE_CAP}) exceeds available features ({len(all_feature_cols)})')

selector_imputer = SimpleImputer(strategy='median')
X_train_selector = selector_imputer.fit_transform(X_train)

feature_selector = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    min_samples_leaf=100,
    class_weight='balanced_subsample',
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
feature_selector.fit(X_train_selector, y_train)

selected_features_ranked = (
    pd.DataFrame({
        'feature': all_feature_cols,
        'selector_importance': feature_selector.feature_importances_,
    })
    .sort_values(['selector_importance', 'feature'], ascending=[False, True])
    .reset_index(drop=True)
)

selected_40_features_df = selected_features_ranked.head(FEATURE_CAP).copy()
selected_40_features_df.insert(0, 'rank', np.arange(1, FEATURE_CAP + 1))
selected_feature_path = MODEL_OUTPUT_DIR / 'selected_40_features.csv'
selected_40_features_df.to_csv(selected_feature_path, index=False)

feature_cols = selected_40_features_df['feature'].tolist()

X_train = train_df[feature_cols]
X_valid = valid_df[feature_cols]
X_test = test_df[feature_cols]

print(f'Selected feature count: {len(feature_cols)} of {len(all_feature_cols)} candidates')
print(f'Rebuilt X_train: {X_train.shape}')
print(f'Rebuilt X_valid: {X_valid.shape}')
print(f'Rebuilt X_test:  {X_test.shape}')
print(f'Saved selected feature list to: {selected_feature_path}')
display(selected_40_features_df.round(6))

---
## Section 7: Models

Logistic Regression is retained as the interpretable baseline and Random Forest remains the selected final model. Both models are trained on the capped 40-feature matrix selected from training data only.

In [ ]:
models = {
    'logistic_regression': Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(
            class_weight='balanced',
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ]),
    'random_forest': Pipeline(steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('model', RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=50,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )),
    ]),
}

models

---
## Section 8: Evaluation Helpers

In [ ]:
def safe_roc_auc(y_true, y_score):
    if pd.Series(y_true).nunique() < 2:
        return np.nan
    return roc_auc_score(y_true, y_score)


def evaluate_predictions(model_name, split_name, y_true, y_score, threshold):
    y_pred = (y_score >= threshold).astype(int)
    return {
        'model': model_name,
        'split': split_name,
        'threshold': threshold,
        'n': len(y_true),
        'positive_rate': float(np.mean(y_true)),
        'predicted_positive_rate': float(np.mean(y_pred)),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'precision_up': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        'recall_up': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_up': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
        'f1_macro': f1_score(y_true, y_pred, average='macro', zero_division=0),
        'roc_auc': safe_roc_auc(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }


def tune_threshold(y_true, y_score):
    thresholds = np.linspace(0.05, 0.95, 181)
    rows = []
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        rows.append({
            'threshold': threshold,
            'f1_up': f1_score(y_true, y_pred, pos_label=1, zero_division=0),
            'precision_up': precision_score(y_true, y_pred, pos_label=1, zero_division=0),
            'recall_up': recall_score(y_true, y_pred, pos_label=1, zero_division=0),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        })
    threshold_df = pd.DataFrame(rows)
    best = threshold_df.sort_values(['f1_up', 'balanced_accuracy'], ascending=False).iloc[0]
    return float(best['threshold']), threshold_df

---
## Section 9: Train And Evaluate

In [ ]:
metric_rows = []
prediction_frames = []
threshold_tables = {}
fitted_models = {}

for model_name, estimator in models.items():
    print(f'Training {model_name}...')
    estimator.fit(X_train, y_train)
    fitted_models[model_name] = estimator

    valid_score = estimator.predict_proba(X_valid)[:, 1]
    best_threshold, threshold_df = tune_threshold(y_valid, valid_score)
    threshold_tables[model_name] = threshold_df
    print(f'  Validation-tuned threshold: {best_threshold:.3f}')

    for split_name, split_df, X_split, y_split in [
        ('valid', valid_df, X_valid, y_valid),
        ('test', test_df, X_test, y_test),
    ]:
        y_score = estimator.predict_proba(X_split)[:, 1]
        y_pred = (y_score >= best_threshold).astype(int)
        metric_rows.append(evaluate_predictions(model_name, split_name, y_split, y_score, best_threshold))

        pred = split_df[['Date', 'target_date', 'Ticker', 'sector', 'split', 'target']].copy()
        pred['model'] = model_name
        pred['y_score'] = y_score
        pred['y_pred'] = y_pred
        prediction_frames.append(pred)

metrics_df = pd.DataFrame(metric_rows)
predictions_df = pd.concat(prediction_frames, ignore_index=True)

metrics_path = MODEL_OUTPUT_DIR / 'model_metrics.csv'
predictions_path = MODEL_OUTPUT_DIR / 'model_predictions.parquet'
metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_parquet(predictions_path, index=False)

print(f'Saved metrics to: {metrics_path}')
print(f'Saved predictions to: {predictions_path}')
display(metrics_df.sort_values(['split', 'f1_up'], ascending=[True, False]).round(4))

---
## Section 10: Final Model Selection

Random Forest is selected as the final model and Logistic Regression remains in the notebook as an interpretable baseline. After the 40-feature cap is applied, rerun the training and evaluation cells so the saved metrics, thresholds, and prediction files reflect the capped feature matrix.

The final model is still not a standalone trading rule: precision and predicted-`Up` rates should be reviewed after rerun, and outputs should be treated as a screening signal for further analysis.

In [ ]:
FINAL_MODEL = 'random_forest'

final_model_metrics = metrics_df[
    (metrics_df['model'] == FINAL_MODEL) & (metrics_df['split'] == 'test')
].copy()
if final_model_metrics.empty:
    raise ValueError(f'No test metrics found for final model: {FINAL_MODEL}')

final_model_predictions = predictions_df[
    (predictions_df['model'] == FINAL_MODEL) & (predictions_df['split'] == 'test')
].copy()
if final_model_predictions.empty:
    raise ValueError(f'No test predictions found for final model: {FINAL_MODEL}')

final_metrics_path = MODEL_OUTPUT_DIR / 'final_model_metrics.csv'
rf_metrics_path = MODEL_OUTPUT_DIR / 'random_forest_test_metrics.csv'
rf_predictions_path = MODEL_OUTPUT_DIR / 'random_forest_test_predictions.parquet'

final_model_metrics.to_csv(final_metrics_path, index=False)
final_model_metrics.to_csv(rf_metrics_path, index=False)
final_model_predictions.to_parquet(rf_predictions_path, index=False)

print(f'Selected final model: {FINAL_MODEL}')
print(f'Saved final model metrics to: {final_metrics_path}')
print(f'Saved Random Forest test metrics to: {rf_metrics_path}')
print(f'Saved Random Forest test predictions to: {rf_predictions_path}')

display(final_model_metrics.round(4))

rf_row = final_model_metrics.iloc[0]
print(
    'Random Forest test summary: '
    f"F1 Up={rf_row['f1_up']:.3f}, "
    f"ROC-AUC={rf_row['roc_auc']:.3f}, "
    f"PR-AUC={rf_row['pr_auc']:.3f}, "
    f"MCC={rf_row['mcc']:.3f}, "
    f"predicted-Up rate={rf_row['predicted_positive_rate']:.1%}"
)

---
## Section 11: Confusion Matrices And Curves

In [ ]:
test_metrics = metrics_df[metrics_df['split'] == 'test'].copy()
fig, axes = plt.subplots(1, len(models), figsize=(6 * len(models), 5))
if len(models) == 1:
    axes = [axes]

for ax, model_name in zip(axes, models.keys()):
    threshold = test_metrics.loc[test_metrics['model'] == model_name, 'threshold'].iloc[0]
    pred = predictions_df[(predictions_df['model'] == model_name) & (predictions_df['split'] == 'test')]
    ConfusionMatrixDisplay.from_predictions(
        pred['target'],
        pred['y_pred'],
        display_labels=['Down/Flat', 'Up'],
        normalize='true',
        cmap='Blues',
        ax=ax,
        values_format='.2f',
    )
    ax.set_title(f'{model_name}\nthreshold={threshold:.3f}')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for model_name in models.keys():
    pred = predictions_df[(predictions_df['model'] == model_name) & (predictions_df['split'] == 'test')]
    RocCurveDisplay.from_predictions(pred['target'], pred['y_score'], name=model_name, ax=axes[0])
    PrecisionRecallDisplay.from_predictions(pred['target'], pred['y_score'], name=model_name, ax=axes[1])

axes[0].set_title('Test ROC Curve')
axes[1].set_title('Test Precision-Recall Curve')
plt.tight_layout()
plt.show()

---
## Section 12: Sector And Time Slice Checks

In [ ]:
def grouped_binary_metrics(frame, group_cols):
    rows = []
    for key, group in frame.groupby(group_cols):
        if not isinstance(key, tuple):
            key = (key,)
        y_true = group['target'].astype(int)
        y_pred = group['y_pred'].astype(int)
        y_score = group['y_score']
        row = {col: val for col, val in zip(group_cols, key)}
        row.update({
            'n': len(group),
            'positive_rate': y_true.mean(),
            'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
            'precision_up': precision_score(y_true, y_pred, zero_division=0),
            'recall_up': recall_score(y_true, y_pred, zero_division=0),
            'f1_up': f1_score(y_true, y_pred, zero_division=0),
            'roc_auc': safe_roc_auc(y_true, y_score),
        })
        rows.append(row)
    return pd.DataFrame(rows)

analysis_df = predictions_df[predictions_df['split'] == 'test'].copy()
analysis_df['year'] = analysis_df['target_date'].dt.year
analysis_df['quarter'] = analysis_df['target_date'].dt.to_period('Q').astype(str)

for model_name in models.keys():
    print(f'\nSector metrics: {model_name}')
    display(grouped_binary_metrics(analysis_df[analysis_df['model'] == model_name], ['sector']).round(4))

    print(f'Quarter metrics: {model_name}')
    display(grouped_binary_metrics(analysis_df[analysis_df['model'] == model_name], ['quarter']).round(4))

---
## Section 13: Feature Importance

These interpretation tables use the final capped 40-feature model matrix, not the full 77-feature candidate set.

In [ ]:
# Importance tables now describe the final 40-feature model matrix.
if len(feature_cols) != FEATURE_CAP:
    raise ValueError(f'Expected {FEATURE_CAP} selected features, found {len(feature_cols)}')

importance_frames = []

if 'random_forest' in fitted_models:
    rf = fitted_models['random_forest'].named_steps['model']
    importance_frames.append(pd.DataFrame({
        'model': 'random_forest',
        'feature': feature_cols,
        'importance': rf.feature_importances_,
    }))

if 'logistic_regression' in fitted_models:
    lr = fitted_models['logistic_regression'].named_steps['model']
    importance_frames.append(pd.DataFrame({
        'model': 'logistic_regression',
        'feature': feature_cols,
        'importance': np.abs(lr.coef_[0]),
    }))

feature_importance_df = pd.concat(importance_frames, ignore_index=True)
importance_path = MODEL_OUTPUT_DIR / 'feature_importance.csv'
feature_importance_df.to_csv(importance_path, index=False)

for model_name in feature_importance_df['model'].unique():
    print(f'\nTop selected features: {model_name}')
    display(feature_importance_df[feature_importance_df['model'] == model_name]
            .sort_values('importance', ascending=False)
            .head(15)
            .round(6))

print(f'Saved 40-feature importance table to: {importance_path}')

---
## Section 14: Final Model Feature Importance

This section exports the Random-Forest-only importance table for the final model, limited to the selected 40 features.

In [ ]:
rf_importance = feature_importance_df[
    feature_importance_df['model'] == FINAL_MODEL
].copy()
if rf_importance.empty:
    raise ValueError('Random Forest feature importance was not found')

rf_importance = rf_importance.sort_values('importance', ascending=False).reset_index(drop=True)
rf_importance_path = MODEL_OUTPUT_DIR / 'random_forest_feature_importance.csv'
rf_importance.to_csv(rf_importance_path, index=False)

print(f'Saved Random Forest feature importance to: {rf_importance_path}')
display(rf_importance.head(20).round(6))